###Imports

In [0]:
from pyspark.sql.functions import (
    col, to_date, trim, upper, lower, initcap,
    concat, concat_ws, lit, when, coalesce, current_timestamp,
    year, month, dayofmonth, dayofweek, dayofyear,
    quarter, weekofyear, date_format, floor,
    datediff, months_between, round as spark_round,
    regexp_replace, expr
)
from pyspark.sql.types import (
    IntegerType, DoubleType, DateType
)
import dlt

In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType, DoubleType

# --- UTILITY: Quarantine Filter Logic ---
def quarantine_filter(df, rules):
    """
    Splits a DataFrame into 'clean' and 'quarantine' based on SQL rules.
    Adds metadata columns for debugging.
    """
    valid_condition = " AND ".join(f"({condition})" for condition in rules.values())
    invalid_condition = f"NOT ({valid_condition})"
    
    clean_df = df.filter(expr(valid_condition))
    
    quarantine_df = (
        df.filter(expr(invalid_condition))
        .withColumn("_quarantine_timestamp", current_timestamp())
        .withColumn("_quarantine_reasons", 
            concat_ws(", ", *[when(~expr(cond), lit(name)) for name, cond in rules.items()])
        )
    )
    return clean_df, quarantine_df

In [0]:
# --- 1. Batch Rules ---
TRANSACTION_RULES = {
    "valid_product": "product_id IS NOT NULL",
    "valid_customer": "customer_id IS NOT NULL",
    "valid_store": "store_id IS NOT NULL",
    "positive_qty": "quantity > 0",
    "valid_date": "transaction_date IS NOT NULL"
}

RETURNS_RULES = {
    "valid_product": "product_id IS NOT NULL",
    "valid_store": "store_id IS NOT NULL",
    "positive_qty": "quantity > 0",
    "valid_date": "return_date IS NOT NULL"
}

# --- 2. Streaming Rules ---
KAFKA_ORDERS_RULES = {
    "has_order_id": "order_id IS NOT NULL",
    "positive_qty": "quantity > 0",
    "positive_amt": "total_amount > 0"
}

KAFKA_INVENTORY_RULES = {
    "valid_product": "product_id IS NOT NULL",
    "valid_store": "store_id IS NOT NULL",
    "valid_stock": "current_stock >= 0",
    "has_change_type": "change_type IS NOT NULL"
}

# --- 3. Master Data Rules ---
PRODUCTS_RULES = {
    "valid_name": "product_name IS NOT NULL",
    "valid_price": "product_retail_price > 0",
    "valid_cost": "product_cost > 0"
}

###Silver Transactions + Quarantine


In [0]:
@dlt.table(
    name="maven_catalog.silver_schema.slv_transactions",
    comment="Cleansed transactions (Streaming)",
    table_properties={"quality": "silver"}
)
@dlt.expect_all_or_drop(TRANSACTION_RULES) # <--- Fix: Visible Metrics
def slv_transactions():
    raw = (
        # FIX 1: Use read_stream (Correct Syntax)
        dlt.read_stream("maven_catalog.bronze_schema.brz_transactions")
        
        .withColumn("transaction_date", to_date(col("transaction_date"), "M/d/yyyy"))
        .withColumn("stock_date", to_date(col("stock_date"), "M/d/yyyy"))
        .withColumn("product_id", col("product_id").cast(IntegerType()))
        .withColumn("customer_id", col("customer_id").cast(IntegerType()))
        .withColumn("store_id", col("store_id").cast(IntegerType()))
        .withColumn("quantity", col("quantity").cast(IntegerType()))
        
        # FIX 2: Create a Dummy Timestamp for Watermarking
        # Streaming deduplication REQUIRES a timestamp watermark.
        # We cast the date to a timestamp (defaults to midnight) so Spark knows when to clear memory.
        .withColumn("watermark_ts", to_timestamp(col("transaction_date"), "M/d/yyyy"))
        .withWatermark("watermark_ts", "24 hours") 
        
        # FIX 3: Streaming Deduplication
        .dropDuplicates(["transaction_date", "product_id", "customer_id", "store_id", "quantity"])
        
        # Cleanup: Drop the helper column
        .drop("watermark_ts")
        .withColumn("_silver_timestamp", current_timestamp())
    )
    
    clean, _ = quarantine_filter(raw, TRANSACTION_RULES)
    return clean

# --- QUARANTINE TABLE (Streaming) ---
@dlt.table(
    name="maven_catalog.silver_schema.quarantine_transactions",
    comment="Invalid transaction rows",
    table_properties={"quality": "quarantine"}
)
def quarantine_transactions():
    # Fix: Must also be read_stream to match lineage
    raw = (
        dlt.read_stream("maven_catalog.bronze_schema.brz_transactions")
        .withColumn("transaction_date", to_date(col("transaction_date"), "M/d/yyyy"))
        .withColumn("stock_date", to_date(col("stock_date"), "M/d/yyyy"))
        .withColumn("product_id", col("product_id").cast(IntegerType()))
        .withColumn("customer_id", col("customer_id").cast(IntegerType()))
        .withColumn("store_id", col("store_id").cast(IntegerType()))
        .withColumn("quantity", col("quantity").cast(IntegerType()))
    )
    _, quarantined = quarantine_filter(raw, TRANSACTION_RULES)
    return quarantined

###Silver Returns + Quarantine

In [0]:
@dlt.table(
    name="maven_catalog.silver_schema.slv_returns",
    comment="Clean returns. Bad rows -> Quarantine.",
    table_properties={"quality": "silver"}
)
@dlt.expect_all_or_drop(RETURNS_RULES)
def slv_returns():
    raw = (
        dlt.read_stream("maven_catalog.bronze_schema.brz_returns")
        .withColumn("return_date", to_date(col("return_date"), "M/d/yyyy"))
        .withColumn("product_id", col("product_id").cast(IntegerType()))
        .withColumn("store_id", col("store_id").cast(IntegerType()))
        .withColumn("quantity", col("quantity").cast(IntegerType()))
        .dropDuplicates(["return_date", "product_id", "store_id"])
    )
    clean, _ = quarantine_filter(raw, RETURNS_RULES)
    return clean

In [0]:
@dlt.table(
    name="maven_catalog.silver_schema.quarantine_returns",
    comment="Invalid return rows with failure reasons",
    table_properties={"quality": "quarantine"}
)
def quarantine_returns():
    # FIX: Use dlt.read() to fix Lineage and CI/CD
    raw = (
        dlt.read_stream("maven_catalog.bronze_schema.brz_returns")
        .withColumn("return_date", to_date(col("return_date"), "M/d/yyyy"))
        .withColumn("product_id", col("product_id").cast(IntegerType()))
        .withColumn("store_id", col("store_id").cast(IntegerType()))
        .withColumn("quantity", col("quantity").cast(IntegerType()))
    )
    # Note: Ensure RETURNS_RULES is defined in the notebook before this cell
    _, quarantined = quarantine_filter(raw, RETURNS_RULES)
    return quarantined

###Products + Quarantine

In [0]:
# --- RULES ---
PRODUCTS_RULES = {
    "null_product_id": "product_id IS NOT NULL",
    "null_product_name": "product_name IS NOT NULL",
    "invalid_price": "product_retail_price > 0",
    "invalid_cost": "product_cost > 0",
    "price_gte_cost": "product_retail_price >= product_cost",
}

# --- SILVER PRODUCT TABLE (Streaming) ---
@dlt.table(
    name="maven_catalog.silver_schema.slv_products",
    comment="Cleansed products with profit margin",
    table_properties={"quality": "silver"}
)
@dlt.expect_all_or_drop(PRODUCTS_RULES)
def slv_products():
    # FIX: Use read_stream for Incremental Processing (Golden Standard)
    raw = (
        dlt.read_stream("maven_catalog.bronze_schema.brz_products_mongo_dlt")
        .withColumn("product_id", col("product_id").cast(IntegerType()))
        .withColumn("product_retail_price", col("product_retail_price").cast(DoubleType()))
        .withColumn("product_cost", col("product_cost").cast(DoubleType()))
        .withColumn("product_weight", col("product_weight").cast(DoubleType()))
        .withColumn("recyclable", coalesce(col("recyclable").cast(IntegerType()), lit(0)))
        .withColumn("low_fat", coalesce(col("low_fat").cast(IntegerType()), lit(0)))
        
        # Calculations
        .withColumn("profit_margin", 
            spark_round((col("product_retail_price") - col("product_cost")) / col("product_retail_price") * 100, 2))
        .withColumn("profit_amount", 
            spark_round(col("product_retail_price") - col("product_cost"), 2))
        
        # Case Statement
        .withColumn("price_tier",
            when(col("product_retail_price") < 2, "Budget")
            .when(col("product_retail_price") < 5, "Mid-Range")
            .when(col("product_retail_price") < 10, "Premium")
            .otherwise("Luxury"))
            
        .withColumn("product_brand", trim(col("product_brand")))
        .withColumn("product_name", trim(col("product_name")))
    )
    
    clean, _ = quarantine_filter(raw, PRODUCTS_RULES)
    return clean

# --- QUARANTINE TABLE (Streaming) ---
@dlt.table(
    name="maven_catalog.silver_schema.quarantine_products",
    comment="Invalid product rows",
    table_properties={"quality": "quarantine"}
)
def quarantine_products():
    # FIX: Must also be read_stream to match the primary table
    raw = (
        dlt.read_stream("maven_catalog.bronze_schema.brz_products_mongo_dlt")
        .withColumn("product_id", col("product_id").cast(IntegerType()))
        .withColumn("product_retail_price", col("product_retail_price").cast(DoubleType()))
        .withColumn("product_cost", col("product_cost").cast(DoubleType()))
    )
    _, quarantined = quarantine_filter(raw, PRODUCTS_RULES)
    return quarantined

###Silver Kafka Orders + Quarantine


In [0]:
# --- TRANSFORM FUNCTION ---
# Accept the dataframe 'df' as an argument so we can pass the stream into it
def _transform_kafka_orders(df):
    return (
        df.withColumn("order_id", trim(col("order_id")))
        .withColumn("total_amount", col("total_amount").cast(DoubleType()))
        .withColumn("product_id", col("product_id").cast(IntegerType()))
        .withColumn("customer_id", col("customer_id").cast(IntegerType()))
        .withColumn("store_id", col("store_id").cast(IntegerType()))
        .withColumn("quantity", col("quantity").cast(IntegerType()))
        .withColumn("event_time", col("event_time").cast("timestamp"))
        
        # GOLDEN STANDARD: Streaming Deduplication requires Watermark
        .withWatermark("event_time", "10 minutes")
        .dropDuplicates(["order_id"])
        
        # Unit price & Size Categories
        .withColumn("unit_price", round(col("total_amount") / col("quantity"), 2))
        .withColumn(
            "order_size",
            when(col("quantity") == 1, "Single")
            .when(col("quantity") <= 3, "Small")
            .when(col("quantity") <= 6, "Medium")
            .otherwise("Large")
        )
    )

# --- SILVER TABLE (Real-Time) ---
@dlt.table(
    name="maven_catalog.silver_schema.slv_kafka_orders",
    comment="Real-time cleansed orders",
    table_properties={"quality": "silver"}
)
@dlt.expect_all_or_drop(KAFKA_ORDERS_RULES) # <--- CRITICAL: Makes Data Quality Visible
def slv_kafka_orders():
    # FIX: Use dlt.read_stream() to enable Real-Time and fix CI/CD path
    raw_stream = dlt.read_stream("maven_catalog.bronze_schema.brz_kafka_orders")
    
    # Apply transformation
    processed = _transform_kafka_orders(raw_stream)
    
    # Apply Quarantine Logic
    clean, _ = quarantine_filter(processed, KAFKA_ORDERS_RULES)
    return clean

# --- QUARANTINE TABLE ---
@dlt.table(
    name="maven_catalog.silver_schema.quarantine_kafka_orders",
    comment="Invalid Kafka order rows",
    table_properties={"quality": "quarantine"}
)
def quarantine_kafka_orders():
    # Re-read stream for quarantine path
    raw_stream = dlt.read_stream("maven_catalog.bronze_schema.brz_kafka_orders")
    processed = _transform_kafka_orders(raw_stream)
    
    _, quarantined = quarantine_filter(processed, KAFKA_ORDERS_RULES)
    return quarantined

###Inventory + Quarantine

In [0]:
# --- TRANSFORMATION LOGIC ---
def _transform_kafka_inventory(df):
    return (
        df.withColumn("product_id", col("product_id").cast(IntegerType()))
        .withColumn("store_id", col("store_id").cast(IntegerType()))
        .withColumn("current_stock", col("current_stock").cast(IntegerType()))
        .withColumn("quantity_change", col("quantity_change").cast(IntegerType()))
        .withColumn("change_type", trim(col("change_type")))
        .withColumn("event_time", col("event_time").cast("timestamp"))
        
        # GOLDEN STANDARD: Streaming Deduplication
        # Kafka has "at-least-once" delivery, so you MUST dedup or you get ghost inventory.
        .withWatermark("event_time", "10 minutes")
        .dropDuplicates(["store_id", "product_id", "event_time"])
        
        # Business Logic
        .withColumn("is_low_stock", when(col("current_stock") <= 25, True).otherwise(False))
        .withColumn("stock_status",
            when(col("current_stock") == 0, "Out of Stock")
            .when(col("current_stock") <= 25, "Low Stock")
            .when(col("current_stock") <= 50, "Normal")
            .otherwise("Well Stocked")
        )
        .withColumn("is_restock", when(col("change_type") == "restock", True).otherwise(False))
    )

# --- SILVER TABLE (Real-Time) ---
@dlt.table(
    name="maven_catalog.silver_schema.slv_kafka_inventory",
    comment="Real-time cleansed inventory.",
    table_properties={"quality": "silver"}
)
@dlt.expect_all_or_drop(KAFKA_INVENTORY_RULES) # <--- Fixes Data Quality Dashboard
def slv_kafka_inventory():
    # FIX: Use dlt.read_stream() for Real-Time and CI/CD
    raw = _transform_kafka_inventory(dlt.read_stream("maven_catalog.bronze_schema.brz_kafka_inventory"))
    clean, _ = quarantine_filter(raw, KAFKA_INVENTORY_RULES)
    return clean

# --- QUARANTINE TABLE ---
@dlt.table(
    name="maven_catalog.silver_schema.quarantine_kafka_inventory",
    comment="Invalid Kafka inventory rows",
    table_properties={"quality": "quarantine"}
)
def quarantine_kafka_inventory():
    # Fix: Re-read stream for quarantine path
    raw = _transform_kafka_inventory(dlt.read_stream("maven_catalog.bronze_schema.brz_kafka_inventory"))
    _, quarantined = quarantine_filter(raw, KAFKA_INVENTORY_RULES)
    return quarantined

###Silver Customers (SCD Type 2)

In [0]:
@dlt.table(
    name="maven_catalog.silver_schema.customers_clean",
    comment="Intermediate cleaned customers for SCD2"
)
def customers_clean():
    return (
        # FIX: Use dlt.read_stream() to fix CI/CD and Lineage
        dlt.read_stream("maven_catalog.bronze_schema.brz_customers_mongo_dlt")
        
        .withColumn("full_name",
            concat(initcap(col("first_name")), lit(" "),
                   initcap(col("last_name"))))
        .withColumn("birthdate",
            to_date(col("birthdate"), "M/d/yyyy"))
        .withColumn("age",
            floor(months_between(
                current_timestamp(), col("birthdate")
            ) / 12).cast(IntegerType()))
        .withColumn("acct_open_date",
            to_date(col("acct_open_date"), "M/d/yyyy"))
        .withColumn("marital_status",
            when(col("marital_status") == "M", "Married")
            .when(col("marital_status") == "S", "Single")
            .otherwise(col("marital_status")))
        .withColumn("gender",
            when(col("gender") == "M", "Male")
            .when(col("gender") == "F", "Female")
            .otherwise(col("gender")))
        .withColumn("homeowner",
            when(col("homeowner") == "Y", "Yes")
            .when(col("homeowner") == "N", "No")
            .otherwise(col("homeowner")))
        .withColumn("income_band_lower",
            regexp_replace(
                regexp_replace(col("yearly_income"), "[\$K +]", ""),
                " - .*", ""
            ).cast(IntegerType()) * 1000)
        .withColumn("customer_postal_code",
            trim(col("customer_postal_code")))
        .withColumn("customer_country",
            trim(upper(col("customer_country"))))
        .withColumn("customer_id",
            col("customer_id").cast(IntegerType()))
        .withColumn("_silver_timestamp", current_timestamp())
    )

###SCD Type 2 Table Definition:


In [0]:
# 1. Define the Target Table
dlt.create_streaming_table(
    name="maven_catalog.silver_schema.slv_customers",
    comment="Silver customers with SCD Type 2 history tracking",
    table_properties={"quality": "silver"}
)

# 2. Apply SCD Type 2 Logic
dlt.apply_changes(
    target="maven_catalog.silver_schema.slv_customers",
    source="maven_catalog.silver_schema.customers_clean", # Ensure this matches your @dlt.view name
    keys=["customer_id"],
    
    # CRITICAL FIX: Wrap the column name in col()
    sequence_by=col("_silver_timestamp"), 
    
    stored_as_scd_type=2
)

###Silver Stores (Joined with Regions)

###Silver Regions

In [0]:
@dlt.table(
    name="maven_catalog.silver_schema.slv_regions",
    comment="Clean regions dimension.",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_region_id", "region_id IS NOT NULL")
def slv_regions():
    # FIX: Use dlt.read for CI/CD safety
    return (
        dlt.read_stream("maven_catalog.bronze_schema.brz_regions")
        .withColumn("region_id", col("region_id").cast(IntegerType()))
        .withColumn("sales_district", trim(col("sales_district")))
        .withColumn("sales_region", trim(col("sales_region")))
        .dropDuplicates(["region_id"])
    )

###Silver Stores and Regions Join

In [0]:
@dlt.table(
    name="maven_catalog.silver_schema.slv_stores",
    comment="Stores enriched with Region data.",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_store_id", "store_id IS NOT NULL")
@dlt.expect("valid_country", "store_country IN ('USA', 'CANADA', 'MEXICO')")
def slv_stores():
    # 1. Read sources using dlt.read()
    # We drop _rescued_data early to keep the schema clean
    stores = (dlt.read_stream("maven_catalog.bronze_schema.brz_stores")
              .drop("_rescued_data", "_source_system", "_ingestion_timestamp", "source_file"))
    
    regions = (dlt.read("maven_catalog.bronze_schema.brz_regions")
               .drop("_rescued_data", "_source_system", "_ingestion_timestamp", "source_file"))

    return (
        # 2. Join Stores with Regions
        stores
        .join(regions, "region_id", "left")
        
        # 3. Apply Transformations
        .withColumn("store_id", col("store_id").cast(IntegerType()))
        .withColumn("region_id", col("region_id").cast(IntegerType()))
        
        # Dates & Age Calculations
        .withColumn("first_opened_date", to_date(col("first_opened_date"), "M/d/yyyy"))
        .withColumn("last_remodel_date", to_date(col("last_remodel_date"), "M/d/yyyy"))
        .withColumn("store_age_years", floor(months_between(current_timestamp(), col("first_opened_date")) / 12).cast(IntegerType()))
        .withColumn("years_since_remodel", floor(months_between(current_timestamp(), col("last_remodel_date")) / 12).cast(IntegerType()))
        
        # Metrics
        .withColumn("total_sqft", col("total_sqft").cast(IntegerType()))
        .withColumn("grocery_sqft", col("grocery_sqft").cast(IntegerType()))
        .withColumn("non_grocery_sqft", col("total_sqft") - col("grocery_sqft"))
        .withColumn("grocery_pct", round(col("grocery_sqft") / col("total_sqft") * 100, 1))
        
        # String Cleanup
        .withColumn("store_country", trim(upper(col("store_country"))))
        .withColumn("store_name", trim(col("store_name")))
        .withColumn("store_city", trim(col("store_city")))
        .withColumn("store_state", trim(col("store_state")))
        
    )

###Silver Calendar (Full Date Dimension)

In [0]:
@dlt.table(
    name="maven_catalog.silver_schema.slv_calendar",
    comment="Full date dimension derived from calendar dates",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_date", "date IS NOT NULL")
def slv_calendar():
    return (
        # FIX: Use dlt.read() for proper lineage and CI/CD support
        dlt.read_stream("maven_catalog.bronze_schema.brz_calendar")
        .withColumn("date", to_date(col("date"), "M/d/yyyy"))
        # Year / Quarter / Month / Week / Day
        .withColumn("year", year(col("date")))
        .withColumn("quarter", quarter(col("date")))
        .withColumn("month", month(col("date")))
        .withColumn("month_name", date_format(col("date"), "MMMM"))
        .withColumn("month_short", date_format(col("date"), "MMM"))
        .withColumn("week_of_year", weekofyear(col("date")))
        .withColumn("day_of_month", dayofmonth(col("date")))
        .withColumn("day_of_week", dayofweek(col("date")))
        .withColumn("day_of_year", dayofyear(col("date")))
        .withColumn("day_name", date_format(col("date"), "EEEE"))
        .withColumn("day_short", date_format(col("date"), "EEE"))
        # Flags
        .withColumn(
            "is_weekend",
            when(dayofweek(col("date")).isin(1, 7), True).otherwise(False)
        )
        .withColumn(
            "is_weekday",
            when(dayofweek(col("date")).isin(1, 7), False).otherwise(True)
        )
        # Fiscal year (assuming Jul-Jun fiscal year)
        .withColumn(
            "fiscal_year",
            when(month(col("date")) >= 7, year(col("date")) + 1)
            .otherwise(year(col("date")))
        )
        .withColumn(
            "fiscal_quarter",
            when(month(col("date")).isin(7,8,9), 1)
            .when(month(col("date")).isin(10,11,12), 2)
            .when(month(col("date")).isin(1,2,3), 3)
            .otherwise(4)
        )
        # Year-Month key for sorting
        .withColumn("year_month", date_format(col("date"), "yyyy-MM"))
        .dropDuplicates(["date"])

    )